# BA283 LSTM training on Colab T4

Single-conditioning (FiLM) `LSTM(1,32)+FiLM+Dense(1)` train on the real gain-conditioned dataset `gdrive:ba283_spice_gain` (452 sims), pulled via rclone (no FUSE mount).

Runtime -> Change runtime type -> **T4 GPU**, then run cells top to bottom.

1. Install rclone, upload your `rclone.conf` (holds the Drive token, written straight to disk).
2. Clone the repo for the `train` package + deps.
3. rclone-copy `ba283_spice_gain` to local disk and remap manifests to `/content`.
4. Train FiLM LSTM (200 epochs, lr 5e-3), export `ba283_lstm32.json`, download it + `best.pth`.

Drop the downloaded `ba283_lstm32.json` into the repo at `plugin/weights/ba283_lstm32.json` to replace the placeholder.

In [ ]:
import os, subprocess
from google.colab import files

# install rclone (no-op if present)
if subprocess.run(['which','rclone'],capture_output=True).returncode:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash', shell=True, check=True)

# upload rclone.conf (token written to disk, not echoed)
cfg = os.path.expanduser('~/.config/rclone/rclone.conf')
os.makedirs(os.path.dirname(cfg), exist_ok=True)
up = files.upload()  # pick your rclone.conf
assert up, 'no file uploaded'
open(cfg,'wb').write(next(iter(up.values())))
REMOTE = subprocess.run(['rclone','listremotes'],capture_output=True,text=True).stdout.split()[0]
print('remote:', REMOTE)

In [ ]:
import getpass, subprocess
TOKEN = getpass.getpass('GitHub token: ')
subprocess.run(['git','clone','-q','https://x-access-token:%s@github.com/gustavokch/ba283-vst3.git'%TOKEN,'/content/ba283'])
%cd /content/ba283
!git pull -q
!pip -q install soundfile pandas tensorboard scipy
import torch
assert torch.cuda.is_available(), 'No GPU: set Runtime type to T4 GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
from train.stage import stage_rclone, remap_manifests
DATA = str(stage_rclone(REMOTE + 'ba283_spice_gain', '/content/dataset'))
M = remap_manifests('/content/dataset', out_dir='/content')
print('staged ->', DATA)
print(M)

In [ ]:
!python -m train.train \
  --train-manifest /content/train.csv --val-manifest /content/val.csv \
  --epochs 200 --lr 5e-3 --hidden 32 --device cuda --conditioning film \
  --checkpoint-dir ckpt --log-dir logs --patience 999 --seed 42

In [ ]:
from pathlib import Path
import torch
from train.model import BA283LSTM
from train.export import export_rtneural_json

m = BA283LSTM(hidden=32, conditioning='film')
m.load_state_dict(torch.load('ckpt/best.pth', map_location='cpu', weights_only=True))
m.eval()
meta = {
    'source': 'colab-t4',
    'epochs': 200,
    'dataset': 'ba283_spice_gain (452 sims, gain-conditioned)',
    'hidden': 32,
    'conditioning': 'film',
}
export_rtneural_json(m, Path('ba283_lstm32.json'), meta)
print('exported', Path('ba283_lstm32.json').stat().st_size, 'bytes')

In [ ]:
from google.colab import files
files.download('ba283_lstm32.json')
files.download('ckpt/best.pth')
files.download('ckpt/last.pth')